# Asset Pricing Tests for Size and Momentum Portfolios

In this notebook, we will replicate the relevant parts of Tables 6 and 7 from Fama and French (2012).

This analysis is equivalent to the analysis in the previous notebook. The main difference is the portfolio dataset.

The previous notebook used portfolios formed using:

- Size
- Book-to-market

This notebook uses portfolios formed using:

- Size
- Momentum

Table 6 reports overall model performance across the portfolios.

Table 7 reports the alpha and alpha t-statistic for each individual portfolio.

The paper reports only the four-factor model.

In [27]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm

DATA_DIR = Path("cleaned_data")

# 1. Load Factor Data
developed_factors = pd.read_csv(
    DATA_DIR / "developed_3_factors.csv", parse_dates=["date"]
)
developed_mom = pd.read_csv(
    DATA_DIR / "developed_momentum.csv", parse_dates=["date"]
)
developed_factors = developed_factors.merge(
    developed_mom, on="date", validate="one_to_one"
)

japan_factors = pd.read_csv(
    DATA_DIR / "japan_3_factors.csv", parse_dates=["date"]
)
japan_mom = pd.read_csv(
    DATA_DIR / "japan_momentum.csv", parse_dates=["date"]
)
japan_factors = japan_factors.merge(
    japan_mom, on="date", validate="one_to_one"
)

# 2. Load Momentum Portfolio Data
developed_portfolios = pd.read_csv(
    DATA_DIR / "developed_25_size_momentum.csv", parse_dates=["date"]
)
japan_portfolios = pd.read_csv(
    DATA_DIR / "japan_25_size_momentum.csv", parse_dates=["date"]
)

all_portfolios = developed_portfolios.columns.drop("date").tolist()
without_microcaps = all_portfolios[
    5:
]  # drops the first size quintile (microcaps)


# 3. Define Analysis Function for Table 6 (Four-Factor Model Focus)
def run_table6_analysis(portfolio_df, factor_df, portfolio_cols):
  merged = pd.merge(portfolio_df, factor_df, on="date", how="inner")
  T = len(merged)
  N = len(portfolio_cols)
  factor_cols = ["Mkt-RF", "SMB", "HML", "WML"]

  alphas = []
  residuals = []
  r2s = []
  se_alphas = []

  X = sm.add_constant(merged[factor_cols])
  F_mean = merged[factor_cols].mean().values
  F_cov = merged[factor_cols].cov().values

  for p in portfolio_cols:
    y = merged[p] - merged["RF"]
    model = sm.OLS(y, X).fit()
    alphas.append(model.params["const"])
    residuals.append(model.resid)
    r2s.append(model.rsquared_adj)
    se_alphas.append(model.bse["const"])

  alphas = np.array(alphas)
  residuals_df = pd.DataFrame(residuals).T
  Sigma = residuals_df.cov().values
  K = len(factor_cols)

  try:
    inv_Sigma = np.linalg.inv(Sigma)
    term1 = (T - N - K) / N
    term2 = 1.0 / (1.0 + np.dot(F_mean.T, np.dot(np.linalg.inv(F_cov), F_mean)))
    term3 = np.dot(alphas.T, np.dot(inv_Sigma, alphas))
    grs_stat = term1 * term2 * term3
  except:
    grs_stat = np.nan

  try:
    sr_a = np.sqrt(np.dot(alphas.T, np.dot(inv_Sigma, alphas)))
  except:
    sr_a = np.nan

  result_dict = {
      "Model": "Four-factor",
      "GRS": round(grs_stat, 2),
      "|a|": round(np.mean(np.abs(alphas)), 2),
      "R2": round(np.mean(r2s), 2),
      "s(a)": round(np.mean(se_alphas), 2),
      "SR(a)": round(sr_a, 2),
  }
  return pd.DataFrame([result_dict]).set_index("Model")


print("Setup and helper functions loaded successfully for Table 6 & 7!")

Setup and helper functions loaded successfully for Table 6 & 7!


In [28]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

In [29]:
DATA_DIR = Path("cleaned_data")

## Load the factor data

The factor data are the same as those used in the previous notebook.

The three-factor files contain:

- `Mkt-RF`
- `SMB`
- `HML`
- `RF`

The momentum files contain:

- `WML`

In [30]:
developed_factors = pd.read_csv(
    DATA_DIR / "developed_3_factors.csv",
    parse_dates=["date"]
)

In [31]:
developed_momentum = pd.read_csv(
    DATA_DIR / "developed_momentum.csv",
    parse_dates=["date"]
)

In [32]:
japan_factors = pd.read_csv(
    DATA_DIR / "japan_3_factors.csv",
    parse_dates=["date"]
)

In [33]:
japan_momentum = pd.read_csv(
    DATA_DIR / "japan_momentum.csv",
    parse_dates=["date"]
)

## Load the size and momentum portfolios

The portfolio files contain value-weighted monthly returns for 25 portfolios.

The portfolios are formed using:

- Five company-size groups
- Five past-return groups

This gives 5 × 5 = 25 portfolios.

The past-return groups range from past losers to past winners.

In [34]:
developed_portfolios = pd.read_csv(
    DATA_DIR / "developed_25_size_momentum.csv",
    parse_dates=["date"]
)

In [35]:
japan_portfolios = pd.read_csv(
    DATA_DIR / "japan_25_size_momentum.csv",
    parse_dates=["date"]
)

## Add the momentum factor

Merge the three-factor and momentum files using `date`.

The `one_to_one` check confirms that each month appears only once in each dataset.

In [36]:
developed_factors = developed_factors.merge(
    developed_momentum,
    on="date",
    validate="one_to_one"
)

In [37]:
japan_factors = japan_factors.merge(
    japan_momentum,
    on="date",
    validate="one_to_one"
)

## Check the datasets

The sample runs from November 1990 to March 2011.

Each dataset should contain 245 monthly observations.

The factor datasets should contain six columns.

The portfolio datasets should contain one date column and 25 portfolio return columns.

In [38]:
print("Developed factors:", developed_factors.shape)
print("Japanese factors:", japan_factors.shape)
print("Developed portfolios:", developed_portfolios.shape)
print("Japanese portfolios:", japan_portfolios.shape)

Developed factors: (245, 6)
Japanese factors: (245, 6)
Developed portfolios: (245, 26)
Japanese portfolios: (245, 26)


## Select the portfolios

The 5x5 results use all 25 portfolios.

The 4x5 results remove the five portfolios in the smallest size group.

The first five portfolio columns belong to the smallest size group.

In [39]:
all_portfolios = developed_portfolios.columns.drop("date").tolist()

In [40]:
without_microcaps = all_portfolios[5:]

In [41]:
print("Number of 5x5 portfolios:", len(all_portfolios))
print("Number of 4x5 portfolios:", len(without_microcaps))

Number of 5x5 portfolios: 25
Number of 4x5 portfolios: 20


## Define the model

Tables 6 and 7 report only the four-factor model.

The four-factor model uses:

- `Mkt-RF`
- `SMB`
- `HML`
- `WML`

In [42]:
models = {
    "Four-factor": ["Mkt-RF", "SMB", "HML", "WML"]
}

## Table 6

Table 6 is equivalent to Table 3 in the previous notebook.

The dependent variables are now the excess returns of the size and momentum portfolios.

For each portfolio:

1. Subtract `RF` from the portfolio return.
2. Run the four-factor regression.
3. Store the fitted regression result.

For the 5x5 cases, Table 6 reports:

- `GRS`
- `|a|`
- `Adjusted R2`
- `s(a)`
- `SR(a)`

These results use all 25 portfolios.

For the 4x5 cases, Table 6 reports only:

- `GRS`
- `|a|`
- `SR(a)`

These results use the 20 portfolios remaining after the smallest size group is removed.

The paper does not report `Adjusted R2` or `s(a)` for the 4x5 cases.

## Task 1: Global portfolios with Global factors, 5x5

Use:

- `developed_portfolios`
- `developed_factors`
- All 25 portfolios in `all_portfolios`

Run a separate four-factor regression for every portfolio.

Store the regression outputs and calculate the five Table 6 statistics.

Print the result for the four-factor model.

Compare your values with the Global 5x5 part of Table 6. State whether the values match and briefly interpret the result.

In [43]:
# Write your code here
# Task 1: Global 5x5 with Global Factors (Table 6)
results_t6_global_5x5 = run_table6_analysis(
    developed_portfolios, developed_factors, all_portfolios
)
display(results_t6_global_5x5)


,GRS,|a|,R2,s(a),SR(a)
Model,,,,,
Four-factor,3.85,0.14,0.94,0.08,0.7


## Task 2: Global portfolios with Global factors, 4x5

Use:

- `developed_portfolios`
- `developed_factors`
- The 20 portfolios in `without_microcaps`

Run a separate four-factor regression for every portfolio.

Store the regression outputs.

Report the three statistics shown in the 4x5 part of Table 6:

- `GRS`
- `|a|`
- `SR(a)`

Compare your values with the Global 4x5 part of Table 6. Discuss whether removing the smallest portfolios improves the model.

In [44]:
# Write your code here
# Task 2: Global 4x5 with Global Factors (Table 6)
results_t6_global_4x5 = run_table6_analysis(
    developed_portfolios, developed_factors, without_microcaps
)
# For 4x5, report GRS, |a|, and SR(a)
display(results_t6_global_4x5[["GRS", "|a|", "SR(a)"]])


,GRS,|a|,SR(a)
Model,,,
Four-factor,2.02,0.09,0.45


## Task 3: Japanese portfolios with Global factors, 5x5

Use:

- `japan_portfolios`
- `developed_factors`
- All 25 portfolios in `all_portfolios`

Run a separate four-factor regression for every Japanese portfolio using Global Developed factors.

Store the regression outputs and calculate the five Table 6 statistics.

Print the result for the four-factor model.

Compare your values with the Japan, Global factors, 5x5 part of Table 6. Interpret how well Global factors explain Japanese size and momentum portfolio returns.

In [45]:
# Write your code here
# Task 3: Japanese 5x5 with Global Factors (Table 6)
results_t6_jp_global_5x5 = run_table6_analysis(
    japan_portfolios, developed_factors, all_portfolios
)
display(results_t6_jp_global_5x5)


,GRS,|a|,R2,s(a),SR(a)
Model,,,,,
Four-factor,1.64,0.62,0.35,0.38,0.46


## Task 4: Japanese portfolios with Global factors, 4x5

Use:

- `japan_portfolios`
- `developed_factors`
- The 20 portfolios in `without_microcaps`

Run a separate four-factor regression for every Japanese portfolio using Global Developed factors.

Store the regression outputs.

Report the three statistics shown in the 4x5 part of Table 6:

- `GRS`
- `|a|`
- `SR(a)`

Compare your values with the Japan, Global factors, 4x5 part of Table 6. Discuss whether removing the smallest Japanese portfolios changes the result.

In [46]:
# Write your code here
# Task 4: Japanese 4x5 with Global Factors (Table 6)
results_t6_jp_global_4x5 = run_table6_analysis(
    japan_portfolios, developed_factors, without_microcaps
)
display(results_t6_jp_global_4x5[["GRS", "|a|", "SR(a)"]])


,GRS,|a|,SR(a)
Model,,,
Four-factor,1.32,0.67,0.36


## Task 5: Japanese portfolios with Japanese factors, 5x5

Use:

- `japan_portfolios`
- `japan_factors`
- All 25 portfolios in `all_portfolios`

Run a separate four-factor regression for every Japanese portfolio using Japanese factors.

Store the regression outputs and calculate the five Table 6 statistics.

Print the result for the four-factor model.

Compare your values with the Japan, Local factors, 5x5 part of Table 6. Compare this result with Task 3.

In [47]:
# Write your code here

# Task 5: Japanese 5x5 with Japanese Factors (Table 6)
results_t6_jp_local_5x5 = run_table6_analysis(
    japan_portfolios, japan_factors, all_portfolios
)
display(results_t6_jp_local_5x5)

,GRS,|a|,R2,s(a),SR(a)
Model,,,,,
Four-factor,1.04,0.11,0.93,0.12,0.35


## Task 6: Japanese portfolios with Japanese factors, 4x5

Use:

- `japan_portfolios`
- `japan_factors`
- The 20 portfolios in `without_microcaps`

Run a separate four-factor regression for every Japanese portfolio using Japanese factors.

Store the regression outputs.

Report the three statistics shown in the 4x5 part of Table 6:

- `GRS`
- `|a|`
- `SR(a)`

Compare your values with the Japan, Local factors, 4x5 part of Table 6.

Discuss:

1. Whether removing the smallest portfolios changes the results.
2. Whether Global or Japanese factors explain Japanese portfolio returns better.

In [48]:
# Write your code here
# Task 6: Japanese 4x5 with Japanese Factors (Table 6)
results_t6_jp_local_4x5 = run_table6_analysis(
    japan_portfolios, japan_factors, without_microcaps
)
display(results_t6_jp_local_4x5[["GRS", "|a|", "SR(a)"]])


,GRS,|a|,SR(a)
Model,,,
Four-factor,0.83,0.09,0.28


# Table 7: Individual Portfolio Alphas

Table 7 is equivalent to Table 4 in the previous notebook.

Table 6 summarizes the results across all 25 or 20 portfolio regressions.

Table 7 reports the alpha and alpha t-statistic for each individual portfolio.

For each portfolio, report:

- `a`: The regression alpha
- `t(a)`: The t-statistic of the alpha

These values are taken directly from the stored regression outputs.

The values are not averaged across portfolios.

## Task 7: Global portfolio alphas using Global factors

You have already estimated and stored the regression outputs for Global portfolios using Global factors in Task 1.

Use the stored 5x5 four-factor regression results.

Report:

- The alpha of each portfolio
- The t-statistic of each alpha

Arrange the 25 alphas in a 5 × 5 matrix.

Arrange the 25 alpha t-statistics in another 5 × 5 matrix.

Compare your results with:

`Global size-momentum returns regressed on global factors`

in Table 7.

Identify any patterns across size and past-return groups.

In [49]:
# Write your code here
# Task 7: Generate 5x5 Alpha and t-stat matrices for Global Size-Momentum portfolios (Table 7)
merged_dev = pd.merge(developed_portfolios, developed_factors, on="date", how="inner")
factor_cols = ["Mkt-RF", "SMB", "HML", "WML"]
X_4f = sm.add_constant(merged_dev[factor_cols])

alphas_list = []
t_list = []

for p in all_portfolios:
  y = merged_dev[p] - merged_dev["RF"]
  model = sm.OLS(y, X_4f).fit()
  alphas_list.append(model.params["const"])
  t_list.append(model.tvalues["const"])

row_labels = ["Small", "2", "3", "4", "Big"]
col_labels = ["Losers", "2", "3", "4", "Winners"]

alpha_matrix = pd.DataFrame(
    np.array(alphas_list).reshape(5, 5), index=row_labels, columns=col_labels
).round(2)
t_matrix = pd.DataFrame(
    np.array(t_list).reshape(5, 5), index=row_labels, columns=col_labels
).round(2)

print("Four-Factor Alphas (% per month) - Global Size-Momentum:")
display(alpha_matrix)
print("t-statistics of Alphas:")
display(t_matrix)


Four-Factor Alphas (% per month) - Global Size-Momentum:


,Losers,2,3,4,Winners
Small,-0.07,0.10,0.19,0.46,0.76
2,-0.05,-0.04,-0.08,0.12,0.29
3,0.09,-0.06,-0.08,-0.15,0.00
4,0.13,-0.04,-0.04,-0.15,0.02
Big,0.19,0.02,-0.09,-0.11,-0.15


t-statistics of Alphas:


,Losers,2,3,4,Winners
Small,-0.64,1.32,2.52,6.02,6.40
2,-0.63,-0.68,-1.07,1.78,3.77
3,1.04,-0.78,-1.14,-2.11,0.00
4,1.28,-0.62,-0.63,-2.16,0.23
Big,1.94,0.32,-1.31,-1.93,-1.65


## Task 8: Japanese portfolio alphas using Japanese factors

You have already estimated and stored the regression outputs for Japanese portfolios using Japanese factors in Task 5.

Use the stored 5x5 four-factor regression results.

Report:

- The alpha of each portfolio
- The t-statistic of each alpha

Arrange the 25 alphas in a 5 × 5 matrix.

Arrange the 25 alpha t-statistics in another 5 × 5 matrix.

Compare your results with:

`Japanese size-momentum returns regressed on Japanese factors`

in Table 7.

In [50]:
# Write your code here

# Task 8: Generate 5x5 Alpha matrices for Japanese Size-Momentum portfolios using Local factors (Table 7)
merged_jp = pd.merge(japan_portfolios, japan_factors, on="date", how="inner")
X_jp_4f = sm.add_constant(merged_jp[factor_cols])

jp_alphas = []
jp_t = []

for p in all_portfolios:
  y = merged_jp[p] - merged_jp["RF"]
  model = sm.OLS(y, X_jp_4f).fit()
  jp_alphas.append(model.params["const"])
  jp_t.append(model.tvalues["const"])

jp_alpha_matrix = pd.DataFrame(
    np.array(jp_alphas).reshape(5, 5), index=row_labels, columns=col_labels
).round(2)
jp_t_matrix = pd.DataFrame(
    np.array(jp_t).reshape(5, 5), index=row_labels, columns=col_labels
).round(2)

print("Local Four-Factor Alphas (% per month) - Japan Size-Momentum:")
display(jp_alpha_matrix)
print("t-statistics of Local Alphas:")
display(jp_t_matrix)

Local Four-Factor Alphas (% per month) - Japan Size-Momentum:


,Losers,2,3,4,Winners
Small,0.31,0.32,0.12,0.24,0.04
2,0.04,-0.02,-0.02,-0.06,-0.03
3,-0.06,-0.21,-0.14,-0.01,0.03
4,0.04,-0.08,-0.14,-0.15,0.05
Big,0.10,-0.19,-0.25,-0.11,0.01


t-statistics of Local Alphas:


,Losers,2,3,4,Winners
Small,2.14,2.54,1.16,2.05,0.21
2,0.34,-0.22,-0.21,-0.51,-0.20
3,-0.51,-2.09,-1.19,-0.11,0.21
4,0.30,-0.69,-1.14,-1.23,0.33
Big,0.63,-1.59,-1.97,-1.06,0.12
